# Episode 03: Supervised Fine-Tuning (SFT) Mechanics & Loss Math

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning, PEFT & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\textbf{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Master in this Notebook
In the previous episode, we established that **SFT (Supervised Fine-Tuning)** is an *objective*—the technique of training an LLM on curated prompt-response pairs $(x, y)$.

In this notebook, we look under the hood of SFT from **absolute mathematical and algorithmic first principles**:
1. **The Base vs. SFT Behavioral Shift:** Why a pre-trained base model autocompletes instead of answering, and how SFT converts an autocomplete engine into an assistant.
2. **From Embeddings to Logits:** Tracing how an LLM converts token sequences into probability distributions over a vocabulary ($V$).
3. **Cross-Entropy Loss From Scratch:** Implementing the exact negative log-likelihood loss by hand in pure PyTorch math and matching `torch.nn.CrossEntropyLoss` to the last decimal place.
4. **The Infamous "Token Shift" Mystery:** Why every causal LM training loop writes `shift_logits = logits[..., :-1, :]` and `shift_labels = labels[..., 1:]`.
5. **The Magic of `ignore_index = -100`:** How PyTorch selectively ignores user prompt tokens so gradients backpropagate strictly through assistant responses.


---

## 0. The Base Model vs. SFT Behavioral Shift

Before writing loss equations, let's understand why SFT is necessary in the first place.

### The Autocomplete Reflex of a Base Model
Foundation models (like base `Llama-3-8B` or `Qwen2.5-7B`) are trained on one simple objective: **Next-Token Prediction on internet text**.

If you give a base model the prompt:
> *"What is the capital of France?"*

A base model does **not** know it is an AI assistant! It sees text that looks like a geography worksheet, and its highest-probability continuation is often:
> *"What is the capital of Germany? What is the capital of Italy?"*

Or if you prompt:
> *"Translate 'Hello' to Spanish:"*

The base model might autocomplete:
> *"Translate 'Goodbye' to Spanish: Translate 'Cat' to Spanish:"*

### What SFT Actually Does
SFT trains the model on structured conversation pairs formatted with special role markers (e.g. ChatML):
```text
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```
Through SFT, the model learns the **behavioral reflex**: when I see `<|im_start|>user`, I wait for `<|im_start|>assistant` and then immediately emit the direct answer.


---

## 1. Setup & Deterministic Environment

We only need standard PyTorch and NumPy. All operations in this notebook are pure tensor mathematics and will execute instantaneously on any CPU or Apple Silicon GPU.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Deterministic seed for reproducible intuition
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Environment ready. Active PyTorch Device: {device}")


---

## 2. From Hidden States to Logits: The Next-Token Distribution

In any causal language model:
1. An input sequence of $T$ tokens is mapped to embeddings of hidden dimension $D$ (e.g. $D = 4096$).
2. The tokens pass through multiple Transformer layers.
3. At the final layer, the hidden state $h_t \in \mathbb{R}^D$ is projected by the **Language Modeling Head (LM Head)** into a vector of size $V$ (the vocabulary size, e.g. $V = 32,000$ or $151,936$):

$$z_t = h_t \cdot W_{\text{vocab}}^T \quad \implies z_t \in \mathbb{R}^V$$

These unnormalized prediction scores $z_t$ are called **Logits**.

To turn logits into probabilities that sum to $1.0$, we apply the **Softmax** function:

$$P(y_t = k) = \frac{e^{z_{t, k}}}{\sum_{j=1}^V e^{z_{t, j}}}$$

Let's test this with a tiny toy vocabulary of 5 words.


In [ ]:
# Tiny vocabulary for demonstration
vocab = {0: "<pad>", 1: "The", 2: "capital", 3: "is", 4: "Paris"}
V = len(vocab)

# Suppose the model outputs raw unnormalized logits for 1 step:
# Higher number = model thinks that word is more likely
sample_logits = torch.tensor([[-1.5, 2.0, 0.5, 1.2, 4.8]]) # Shape: (1, 5)

# 1. Manual Softmax calculation: e^z / sum(e^z)
exp_logits = torch.exp(sample_logits - sample_logits.max()) # subtract max for numerical stability
manual_probs = exp_logits / exp_logits.sum(dim=-1, keepdim=True)

# 2. PyTorch native Softmax
torch_probs = F.softmax(sample_logits, dim=-1)

print("Vocabulary:", vocab)
print("Raw Logits:", sample_logits.numpy().round(2))
print("Probabilities (sum = 1.0):")
for idx, word in vocab.items():
    p = manual_probs[0, idx].item()
    print(f"  Token {idx} ('{word:<7}'): {p * 100:6.2f}%")

print("Exact match between manual and PyTorch softmax?", torch.allclose(manual_probs, torch_probs))


---

## 3. The Cross-Entropy Loss From Absolute Scratch

During SFT, we compare the model's predicted probability distribution $P$ against the **ground-truth target token** $y^*$.

The loss function used across all modern autoregressive models is the **Negative Log-Likelihood (Cross-Entropy Loss)**:

$$\mathcal{L} = -\log P(y^*)$$

### Why Negative Logarithm?
Look at the mathematical behavior of $-\log(P)$:
* If the model predicts the true target token with **$99\%$ confidence** ($P = 0.99$):
  $$\mathcal{L} = -\log(0.99) = \mathbf{0.010} \quad (Near-zero penalty)$$
* If the model predicts the true target token with **$50\%$ confidence** ($P = 0.50$):
  $$\mathcal{L} = -\log(0.50) = \mathbf{0.693} \quad (Moderate penalty)$$
* If the model predicts the true target token with **$1\%$ confidence** ($P = 0.01$):
  $$\mathcal{L} = -\log(0.01) = \mathbf{4.605} \quad (Severe penalty!)$$
* As $P \to 0$, $-\log(P) \to +\infty$. The optimizer forcefully punishes the model when it assigns low probability to the correct token.

Let's compute this step-by-step by hand and compare against `nn.CrossEntropyLoss`.


In [ ]:
# Target word is "Paris" (Token index 4)
target_token_id = torch.tensor([4]) # Shape: (1,)

# Step 1: Get the model's predicted probability for the target token
target_prob = manual_probs[0, target_token_id].item()

# Step 2: Negative Log Likelihood: -log(P)
manual_loss = -math.log(target_prob)

# Step 3: Compute using PyTorch's native nn.CrossEntropyLoss
criterion = nn.CrossEntropyLoss()
pytorch_loss = criterion(sample_logits, target_token_id).item()

print(f"Target Token: Index {target_token_id.item()} ('{vocab[target_token_id.item()]}')")
print(f"Model Predicted Probability: {target_prob * 100:.2f}%")
print(f"Manual Cross-Entropy Loss:   {manual_loss:.6f}")
print(f"PyTorch nn.CrossEntropyLoss: {pytorch_loss:.6f}")
print(f"Exact numerical match?       {math.isclose(manual_loss, pytorch_loss, rel_tol=1e-5)}")


---

## 4. The Infamous "Token Shift" Mystery

When AI engineers write their first custom SFT loop or read Hugging Face `transformers` source code, they always encounter these two mysterious lines:

```python
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()
loss = criterion(shift_logits.view(-1, V), shift_labels.view(-1))
```

> **Why do we slice `[:-1]` on logits and `[1:]` on labels?**

### The Intuition of Next-Token Prediction
An autoregressive language model cannot see the future. At each position $t$, the model takes tokens up to $t$ and **predicts token $t+1$**.

Let's trace a 4-token sequence: `["The", "capital", "is", "Paris"]`

| Position ($t$) | Token Input to Model ($x_t$) | What Model's Output Logit Represents | Ground Truth Label We Evaluate Against ($y_{t+1}$) |
|---|---|---|---|
| **0** | `"The"` | Prediction for the *next* token | `"capital"` |
| **1** | `"capital"` | Prediction for the *next* token | `"is"` |
| **2** | `"is"` | Prediction for the *next* token | `"Paris"` |
| **3** | `"Paris"` | Prediction for the token *after* Paris | *(No ground truth available in this sequence!)* |

Look at the alignment:
- The prediction at position 0 (`logits[0]`) must be compared against the label at position 1 (`labels[1]`).
- The prediction at position 1 (`logits[1]`) must be compared against the label at position 2 (`labels[2]`).
- The prediction at position 2 (`logits[2]`) must be compared against the label at position 3 (`labels[3]`).
- The final logit (`logits[3]`) has **no target**, so we drop it: `logits[:-1]`!
- The first input (`labels[0]`) was the **prompt start**, so no prior token predicted it; we drop it from targets: `labels[1:]`!

Let's test this in code with full tensor inspection.


In [ ]:
# Sequence of 4 token IDs: "The" (1), "capital" (2), "is" (3), "Paris" (4)
input_ids = torch.tensor([[1, 2, 3, 4]]) # Shape: (batch=1, seq_len=4)
labels    = torch.tensor([[1, 2, 3, 4]])

# Simulate model output: (batch=1, seq_len=4, vocab_size=5)
# Let's create mock logits for each position
mock_logits = torch.randn(1, 4, V)

# The Token Shift:
shift_logits = mock_logits[..., :-1, :].contiguous() # Drop last prediction (shape: 1, 3, 5)
shift_labels = labels[..., 1:].contiguous()          # Drop first prompt token (shape: 1, 3)

print("Original input_ids shape: ", input_ids.shape)
print("Original logits shape:    ", mock_logits.shape)
print("After Shift:")
print("shift_logits shape:       ", shift_logits.shape, " (Dropped final position)")
print("shift_labels shape:       ", shift_labels.shape, " (Dropped first position)")
print("Aligned pairs for loss calculation:")
for pos in range(shift_labels.shape[1]):
    pred_step = pos
    target_tok_id = shift_labels[0, pos].item()
    input_tok_id = input_ids[0, pos].item()
    print(f"  Step {pos}: Input '{vocab[input_tok_id]}' -> Model predicts Logits[{pred_step}] -> Evaluated against Target '{vocab[target_tok_id]}'")


---

## 5. Masking the Prompt: The Magic of `ignore_index = -100`

In modern Supervised Fine-Tuning (SFT), a training example consists of:
$$\text{Full Sequence} = [\underbrace{\text{User Prompt}}_{\text{Context}}, \quad \underbrace{\text{Assistant Response}}_{\text{Target Completion}}]$$

### The Fatal Flaw of Naive SFT:
If you compute loss over the entire sequence, you are forcing the model to minimize loss on the **User's prompt** as well as the assistant's reply!
- The model will spend capacity memorizing how users ask questions.
- It leads directly to the **Prompt Parrot Bug** (which we will break and analyze in Episode 07).

### The Solution: PyTorch's `ignore_index = -100`
PyTorch's `CrossEntropyLoss` has a built-in parameter: `ignore_index=-100`.
Any token position in `labels` whose value is `-100` is **completely ignored in the loss calculation and contributes zero gradient**!

Let's test this in action:
Suppose tokens 0 and 1 belong to the User Prompt (`"The capital"`), and tokens 2 and 3 belong to the Assistant Response (`"is Paris"`).
We mask the prompt tokens by replacing their label with `-100`.


In [ ]:
# Target labels: Prompt tokens replaced with -100, response tokens kept intact
masked_labels = torch.tensor([[-100, -100, 3, 4]]) # Shape: (1, 4)

# Shift them as standard:
shift_masked_labels = masked_labels[..., 1:].contiguous() # Shape: (1, 3) -> [-100, 3, 4]

# Unmasked vs Masked Loss
criterion_all    = nn.CrossEntropyLoss()                     # Default ignore_index=-100
criterion_masked = nn.CrossEntropyLoss(ignore_index=-100)

loss_unmasked = criterion_all(shift_logits.view(-1, V), shift_labels.view(-1))
loss_masked   = criterion_masked(shift_logits.view(-1, V), shift_masked_labels.view(-1))

print("Shifted Labels without masking:", shift_labels.numpy())
print("Shifted Labels with masking:   ", shift_masked_labels.numpy())
print(f"Unmasked Loss (computes loss on prompt + answer): {loss_unmasked.item():.4f}")
print(f"Masked Loss   (computes loss ONLY on answer):     {loss_masked.item():.4f}")
print("Notice the difference: Masked loss only optimizes the tokens we want the model to generate!")


---

## 6. Build: A Complete Minimal SFT Step From Scratch

Now let's put all the pieces together into a clean, complete SFT training step:
1. Embed tokens.
2. Forward pass through a mock transformer layer + LM Head.
3. Shift logits and shifted masked labels.
4. Compute Cross-Entropy Loss with `ignore_index=-100`.
5. Backward pass to verify gradients flow **only** to active parameters.


In [ ]:
# Define a tiny causal LM architecture for first-principles demonstration
class MinimalCausalLM(nn.Module):
    def __init__(self, vocab_size=5, hidden_dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.transformer_block = nn.Linear(hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, input_ids):
        # 1. Embed tokens: (B, T) -> (B, T, D)
        h = self.embed(input_ids)
        # 2. Mock causal processing: (B, T, D) -> (B, T, D)
        h = F.relu(self.transformer_block(h))
        # 3. Project to logits: (B, T, D) -> (B, T, V)
        logits = self.lm_head(h)
        return logits

# Initialize toy model and optimizer
model = MinimalCausalLM(vocab_size=V, hidden_dim=8)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

# Training loop: 10 steps of pure SFT
print("Step | SFT Masked Loss")
print("---- | -----------------")
for step in range(1, 11):
    optimizer.zero_grad()
    
    # 1. Forward pass
    logits = model(input_ids)
    
    # 2. Token shift
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = masked_labels[..., 1:].contiguous()
    
    # 3. Compute loss with prompt masking
    loss = F.cross_entropy(shift_logits.view(-1, V), shift_labels.view(-1), ignore_index=-100)
    
    # 4. Backward pass & step
    loss.backward()
    optimizer.step()
    
    if step % 2 == 0 or step == 1:
        print(f"{step:4d} | {loss.item():.6f}")

print("Success! SFT loss decayed smoothly while prompt tokens were 100% ignored.")


---

## 🎯 Episode 03 Comfort Recap

| Concept | Mathematical / PyTorch Operation | Why It Exists |
|---|---|---|
| **Base Model Reflex** | Next-token autocomplete on internet corpus | Causes models to repeat prompt patterns rather than answering. |
| **SFT Objective** | $\mathcal{L} = -\log P(y_{\text{target}})$ | Tunes model probabilities on prompt-response pairs to instill conversational reflexes. |
| **Logits $\to$ Probabilities** | $P = \text{Softmax}(z) = \frac{e^z}{\sum e^z}$ | Normalizes raw layer scores into a valid probability distribution summing to $1.0$. |
| **The Token Shift** | `logits[..., :-1, :]` vs `labels[..., 1:]` | Position $t$ predicts token $t+1$; shifts align model predictions with the true next token. |
| **Prompt Masking** | `labels = -100` (`ignore_index=-100`) | Tells PyTorch to skip prompt tokens so the model only learns to generate the assistant response. |

---

### Up Next:
In **Episode 04: The Alignment & RL Landscape (DPO, GRPO, PPO)**, we explore what happens when SFT is not enough:
- Why SFT only teaches models *what to say*, but cannot teach models *what to avoid*.
- Direct Preference Optimization (DPO): Training on chosen vs. rejected response pairs.
- Group Relative Policy Optimization (GRPO): Verifiable reward scoring without training an expensive critic network (the DeepSeek-R1 paradigm).

👉 Jump into [`comfort-refresh.md`](./comfort-refresh.md) anytime for a 3-minute recap!
